# Import Library

In [1]:
import os
import time
import pandas as pd
import io
from google import genai
from google.genai import types

# Inisialisasi API KEY

In [2]:
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
client = genai.Client()

# Prompt Engineering

In [3]:
print("Meminta Gemini untuk membuat dataset sintetik... (Mohon tunggu sekitar 10-20 detik)")

system_instruction = """Kamu adalah Data Engineer yang bertugas membuat dataset sintetik (dummy) berkualitas tinggi untuk melatih model Machine Learning Text Classification (Multi-Output).
Tugasmu adalah membuat baris data dalam format CSV murni dengan pemisah koma (,).
HANYA BERIKAN OUTPUT DALAM FORMAT CSV tanpa markdown, tanpa penjelasan, dan dengan header yang tepat. Pastikan teks yang mengandung koma diapit oleh tanda kutip ganda ("").

Kolom yang dibutuhkan:
1. input_teks
2. kategori_aset
3. severity
4. root_cause
5. tindakan

Aturan Kolom 1 (input_teks):
Berisi gabungan kalimat keluhan orang awam (bahasa santai/panik) dan kalimat laporan teknisi lapangan (bahasa teknis praktis, kadang ada singkatan/typo). Panjang sekitar 2-3 kalimat.

Aturan Kolom 2-5 (Target Klasifikasi):
Berisi label kategori yang sesuai dan logis berdasarkan teks input.

Contoh Format Output:
input_teks,kategori_aset,severity,root_cause,tindakan
"Tolong dong AC di ruang HRD netes air ke karpet. Udah diservis, ternyata selang pembuangannya mampet banyak lumut, langsung disemprot dan tambah freon.","HVAC","Sedang","Saluran Tersumbat","Pembersihan Saluran"
"Mesin pompa air bawah mati total bau gosong. Kabelnya konslet digigit tikus, udah diganti kabel baru dan dites nyala.","Kelistrikan & Pompa","Tinggi","Kabel Putus/Konslet","Penggantian Komponen"
"""

user_prompt = """Buatkan 15 baris data sintetik baru. 
Gunakan variasi nama mesin industri (Pompa Air, Genset, Conveyor Belt, Kompresor, AC Sentral, Motor Listrik, dll). 
Gunakan variasi tingkat severity (Rendah, Sedang, Tinggi, Kritis). 
Pastikan gaya bahasa pada 'input_teks' sangat bervariasi (ada yang marah, ada yang laporan rutin biasa, ada yang pakai singkatan seperti 'yg', 'udah', 'benerin') agar model AI belajar variasi bahasa natural."""

Meminta Gemini untuk membuat dataset sintetik... (Mohon tunggu sekitar 10-20 detik)


# Processing ke CSV

In [4]:
file_name = "../../data/dataset_ticketing_generator.csv"
jumlah_loop = 10

print(f"🚀 Memulai pembuatan dataset '{file_name}'...\n")

for i in range(jumlah_loop):
    print(f"🔄 Meng-generate batch {i+1} dari {jumlah_loop}...")
    
    try:
        # Memanggil Gemini API
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=user_prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.8, # Temperature 0.8 agar variasi teks keluhannya sangat beragam dan tidak kaku
            )
        )
        
        csv_raw_data = response.text
        
        # Membersihkan formatting markdown (```csv ... ```) jika ada
        if "```csv" in csv_raw_data:
            csv_raw_data = csv_raw_data.replace("```csv", "").replace("```", "").strip()
        elif "```" in csv_raw_data:
            csv_raw_data = csv_raw_data.replace("```", "").strip()
            
        # Membaca teks CSV menjadi Pandas DataFrame
        df_sintetik = pd.read_csv(io.StringIO(csv_raw_data))
        
        # Menyimpan ke file
        if os.path.exists(file_name):
            # Jika file sudah ada, tambahkan ke baris bawahnya (append) tanpa header
            df_sintetik.to_csv(file_name, mode='a', header=False, index=False)
        else:
            # Jika file belum ada, buat baru beserta headernya
            df_sintetik.to_csv(file_name, index=False)
            
        print(f"✅ Batch {i+1} berhasil ditambahkan!")
        
        # Pause 3 detik agar tidak terkena limit API dari Google (Rate Limiting)
        time.sleep(3)
        
    except Exception as e:
        print(f"❌ Gagal memproses batch {i+1}. Error: {e}")
        print("Teks mentah dari Gemini:\n", csv_raw_data)

# 5. Mengecek Hasil Akhir
if os.path.exists(file_name):
    df_total = pd.read_csv(file_name)
    print(f"\n🎉 SELESAI! Dataset Ticketing berhasil dibuat dengan total {df_total.shape[0]} baris.")
    print("\n--- 5 Baris Pertama ---")
    display(df_total.head())

🚀 Memulai pembuatan dataset '../../data/dataset_ticketing_generator.csv'...

🔄 Meng-generate batch 1 dari 10...
✅ Batch 1 berhasil ditambahkan!
🔄 Meng-generate batch 2 dari 10...
✅ Batch 2 berhasil ditambahkan!
🔄 Meng-generate batch 3 dari 10...
✅ Batch 3 berhasil ditambahkan!
🔄 Meng-generate batch 4 dari 10...
✅ Batch 4 berhasil ditambahkan!
🔄 Meng-generate batch 5 dari 10...
✅ Batch 5 berhasil ditambahkan!
🔄 Meng-generate batch 6 dari 10...
✅ Batch 6 berhasil ditambahkan!
🔄 Meng-generate batch 7 dari 10...
✅ Batch 7 berhasil ditambahkan!
🔄 Meng-generate batch 8 dari 10...
✅ Batch 8 berhasil ditambahkan!
🔄 Meng-generate batch 9 dari 10...
✅ Batch 9 berhasil ditambahkan!
🔄 Meng-generate batch 10 dari 10...
❌ Gagal memproses batch 10. Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your 

,input_teks,kategori_aset,severity,root_cause,tindakan
0,"AC di ruang server kok panas banget ya, kayakn...",HVAC,Tinggi,Kebocoran Freon,Penambalan dan Isi Ulang Freon
1,Pompa air di area produksi sering mati sendiri...,Kelistrikan & Pompa,Sedang,Kapasitor Rusak,Penggantian Komponen
2,Genset nggak mau nyala sama sekali pas mati la...,Power Supply & Generator,Kritis,Aki Soak,Penggantian Komponen
3,Conveyor belt di jalur packing suka nyangkut d...,Mekanikal,Sedang,Bearing Aus,Penggantian Komponen
4,"Kompresor angin kok tekanannya drop terus ya, ...",Udara Bertekanan,Tinggi,Seal Rusak,Penggantian Komponen
